***OBJECTIVES:***
 > Predict Ticket Type using:
    ~ Ticket Subject
    ~ Ticket Description

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [2]:
df = pd.read_csv("customer_support_tickets.csv")

In [3]:
df.columns

Index(['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age',
       'Customer Gender', 'Product Purchased', 'Date of Purchase',
       'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status',
       'Resolution', 'Ticket Priority', 'Ticket Channel',
       'First Response Time', 'Time to Resolution',
       'Customer Satisfaction Rating'],
      dtype='str')

In [4]:
features = ['Ticket Subject', 'Ticket Description']
df[features].isna().value_counts()

Ticket Subject  Ticket Description
False           False                 8469
Name: count, dtype: int64

In [18]:
X = df[features]
print(X.shape)
y = df['Ticket Type']
print(y.shape)
print(y.value_counts())

(8469, 2)
(8469,)
Ticket Type
Refund request          1752
Technical issue         1747
Cancellation request    1695
Product inquiry         1641
Billing inquiry         1634
Name: count, dtype: int64


***To Build the Best Model I will build. model to test for accuracy in 3 ways***
    > Normal training with a training and validation set
    > K-Fold Cross-Validation
    > Out-Of-Bag Error
    > Confusion Matrix

In [6]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y,
                                                      train_size=0.8,
                                                      random_state=0)

In [7]:
X_train.head()

,Ticket Subject,Ticket Description
3976,Product recommendation,There seems to be a hardware problem with my {...
4637,Installation support,I'm having an issue with the {product_purchase...
8188,Network problem,"My {product_purchased} crashed, and I lost all..."
185,Software bug,There seems to be a glitch in the {product_pur...
843,Cancellation request,I'm having an issue with the {product_purchase...


In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), features)
    ], remainder='passthrough')

# model = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=0)

results = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, oob_score=True, random_state=0))
])

results.fit(X_train, y_train)
# results.named_steps['model'].oob_score_

print(f"OOB Score: " + str(results.named_steps['model'].oob_score_))
preds = results.predict(X_valid)

OOB Score: 0.20044280442804427


***Check The Model's performance***
    > OOB Score for the first model is 20% indicating a 20% accuracy prediction for new data
    ~ Next cell checks accuracy of the model on the validation set

In [20]:
from sklearn.metrics import classification_report, accuracy_score
print(f"Accuracy Score: {accuracy_score(y_valid, preds)}")
print(classification_report(y_valid, preds))

Accuracy Score: 0.19185360094451004
                      precision    recall  f1-score   support

     Billing inquiry       0.18      0.01      0.02       348
Cancellation request       0.21      0.21      0.21       321
     Product inquiry       0.00      0.00      0.00       342
      Refund request       0.19      0.56      0.29       337
     Technical issue       0.17      0.19      0.18       346

            accuracy                           0.19      1694
           macro avg       0.15      0.19      0.14      1694
        weighted avg       0.15      0.19      0.14      1694



***FINDINGS FROM TRAINING USING FEATURES AS CATEGORICAL VARIABLES***
    ~ Ticket Description is too unique for the model to train effectively; each change in a sentence is treated as a unique Category, leading to a 20% accuracy and )% accuracy in product enquiry

***NEW APPROACH:***
    ~ Switch to TfidfVectorizer to give more "weight" to relevant words also ignore "noisy" words

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer

In [59]:
preprocessor_2 = ColumnTransformer(
    transformers=[
        ('subject_tfidf', TfidfVectorizer(stop_words='english', max_features=500, ngram_range=(1, 2)), 'Ticket Subject'),
        ('desc_tfidf', TfidfVectorizer(stop_words='english', max_features=1000, min_df=5, ngram_range=(1, 2)), 'Ticket Description')
    ],
    remainder='passthrough'
)

In [60]:
results_2 = Pipeline(steps=[
    ('preprocessor', preprocessor_2),
    ('model', RandomForestClassifier(n_estimators=500, oob_score=True, random_state=0, max_depth=20, min_samples_split=5))
])
results_2.fit(X_train, y_train)
preds_2 = results_2.predict(X_valid)

In [61]:
print("OOB Score with TfidfVectorizer: " + str(results_2.named_steps['model'].oob_score_))
print(f"Accuracy Score: {accuracy_score(y_valid, preds_2)}")
print(classification_report(y_valid, preds_2))

OOB Score with TfidfVectorizer: 0.20501845018450185
Accuracy Score: 0.1894923258559622
                      precision    recall  f1-score   support

     Billing inquiry       0.16      0.07      0.09       348
Cancellation request       0.17      0.17      0.17       321
     Product inquiry       0.15      0.08      0.10       342
      Refund request       0.21      0.38      0.27       337
     Technical issue       0.20      0.26      0.22       346

            accuracy                           0.19      1694
           macro avg       0.18      0.19      0.17      1694
        weighted avg       0.18      0.19      0.17      1694



***ANALYSIS***
    > Even with hyperparameter tuning the model accuracy is still at 20% :(
    > Probably overfitting on "irrelevant" words

***TRY:***
    > Try running with GridSearch to run multiple models and find which works best

In [62]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'model__n_estimators': [200, 500],
    'model__max_depth': [20, 50, None],
    'model__min_samples_split': [2, 5, 10],
    'model__max_features': ['sqrt', 'log2']
}
grid_search = GridSearchCV(results_2, param_grid, cv=3, scoring='accuracy', verbose=1)
grid_search.fit(X_train, y_train)

print(f"Best Score: {grid_search.best_score_}")
print(f"Best Params: {grid_search.best_params_}")

Fitting 3 folds for each of 36 candidates, totalling 108 fits
Best Score: 0.20310092504044772
Best Params: {'model__max_depth': 20, 'model__max_features': 'sqrt', 'model__min_samples_split': 2, 'model__n_estimators': 200}


In [63]:
grid_preds = grid_search.predict(X_valid)
print(f"Accuracy Score: {accuracy_score(y_valid, grid_preds)}")
print(classification_report(y_valid, grid_preds))

Accuracy Score: 0.1936245572609209
                      precision    recall  f1-score   support

     Billing inquiry       0.20      0.09      0.13       348
Cancellation request       0.17      0.19      0.18       321
     Product inquiry       0.19      0.11      0.14       342
      Refund request       0.21      0.34      0.26       337
     Technical issue       0.19      0.23      0.21       346

            accuracy                           0.19      1694
           macro avg       0.19      0.19      0.18      1694
        weighted avg       0.19      0.19      0.18      1694



In [67]:
check_df = X_train.copy()
check_df['Actual Label (y)'] = y_train

pd.set_option('display.max_colwidth', None)

check_df[['Ticket Subject', 'Ticket Description', 'Actual Label (y)']].sample(20)

,Ticket Subject,Ticket Description,Actual Label (y)
2268,Network problem,My {product_purchased} is making strange noises and not functioning properly. I suspect there might be a hardware issue. Can you please help me with this? Thank you.\n\n{Product_purchased} is the only product I've noticed that the issue occurs consistently when I use a specific feature or application on my {product_purchased}.,Refund request
5270,Payment issue,"I'm having an issue with the {product_purchased}. Please assist.\n\nThanks for supporting us on Patreon! I've reviewed the troubleshooting steps on the official support website, but they didn't resolve the problem.",Cancellation request
1762,Hardware issue,"I'm facing issues logging into my {product_purchased} account. It says my account is locked. What should I do to unlock it? I've had issues logging into my {product_purchased} account for a while. I've recently updated the firmware of my {product_purchased}, and the issue started happening afterward. Could it be related to the update?",Refund request
714,Data loss,"I'm having an issue with the {product_purchased}. Please assist.\n\nI've been struggling on this card for 4 days now and just don't have much to do with it either.\n\nIt works fine with the same I've followed online tutorials and community forums to troubleshoot the issue, but no luck so far.",Refund request
3035,Installation support,"I'm having an issue with the {product_purchased}. Please assist.\n\nPlease help out by uploading pics on your blog (if possible), so some of your posts will be shared here. I've followed online tutorials and community forums to troubleshoot the issue, but no luck so far.",Product inquiry
2375,Installation support,I'm having an issue with the {product_purchased}. Please assist. I've noticed that the issue occurs consistently when I use a specific feature or application on my {product_purchased}.,Refund request
6972,Data loss,"I'm having an issue with the {product_purchased}. Please assist. — — —\n\nSUBJECT: #2624: the #1248.1248.943_d3b00d36705027 I've recently updated the firmware of my {product_purchased}, and the issue started happening afterward. Could it be related to the update?",Technical issue
3161,Product compatibility,"I'm having an issue with the {product_purchased}. Please assist. I'll be back soon. Thank you for your help I've recently updated the firmware of my {product_purchased}, and the issue started happening afterward. Could it be related to the update?",Technical issue
7011,Product compatibility,"My {product_purchased} crashed, and I lost all the data stored on it. Is there any way to recover the lost data? :o ) This is a lot more complicated than what we are doing here. The first step is I've reviewed the troubleshooting steps on the official support website, but they didn't resolve the problem.",Product inquiry
6689,Product setup,"I'm having an issue with the {product_purchased}. Please assist.\n\nProduct information:\n\nThe product(s) that are mentioned here have not been purchased\n\n(the ""product"" field refers to a product), I've noticed a peculiar error message popping up on my {product_purchased} screen. It says '{error_message}'. What does it mean?",Product inquiry


In [66]:
print(y_train.unique())

<StringArray>
[      'Refund request',      'Product inquiry',      'Technical issue',
      'Billing inquiry', 'Cancellation request']
Length: 5, dtype: str


### _Ticket Type has no correlation with Ticket Subject and Ticket Description making it impossible to build a model to predict Ticket Type from those Features_